# DeepSeek-OCR-2 — Inferencia local (AMD RX 6750 XT)

**Entorno requerido:** `conda activate deepseek-infer`  
**Kernel:** seleccionar el intérprete del entorno `deepseek-infer` (Python 3.11)

Modelo base: `unsloth/DeepSeek-OCR-2`  
Adaptador LoRA: `Lacax/deepseek_original_dataset`  
Backend GPU: `torch-directml` (DirectX 12 → RX 6750 XT)

## Celda 1 — Setup e imports

In [ ]:
import os, sys, json
import torch

# Selección de dispositivo: DirectML (AMD GPU) con fallback a CPU
try:
    import torch_directml
    DEVICE = torch_directml.device()
    print(f"DirectML activo: {DEVICE}  (RX 6750 XT)")
except ImportError:
    DEVICE = torch.device("cpu")
    torch.set_num_threads(os.cpu_count())
    print(f"WARN: DirectML no disponible. Usando CPU ({os.cpu_count()} threads).")

from transformers import AutoTokenizer, AutoModel
from peft import PeftModel

# Rutas — modelos almacenados en F:\Model_Local_inference\models\
BASE_MODEL   = r"F:\Model_Local_inference\models\deepseek_ocr2_base"
LORA_ADAPTER = r"F:\Model_Local_inference\models\deepseek_ocr2_lora"

# Raíz del repo (para localizar imágenes de test)
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

print("Base model:", BASE_MODEL)
print("LoRA adapter:", LORA_ADAPTER)
print("Paths exist:", os.path.isdir(BASE_MODEL), os.path.isdir(LORA_ADAPTER))

## Celda 2 — Cargar modelo

In [ ]:
print("Cargando modelo base (puede tardar ~1-2 min en la primera carga)...")
model = AutoModel.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

print("Aplicando adaptador LoRA...")
model = PeftModel.from_pretrained(model, LORA_ADAPTER)
model = model.to(DEVICE)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
print("Modelo listo.")

## Celda 3 — Función de inferencia

In [ ]:
PROMPT = (
    "Eres un sistema de OCR especializado en tickets de compra españoles. "
    "Extrae exactamente los siguientes campos en JSON válido sin texto adicional:\n"
    '{\n'
    '  "comercio": "string",\n'
    '  "cif": "string",\n'
    '  "fecha": "YYYY-MM-DD",\n'
    '  "total": number,\n'
    '  "items": [{"cantidad": number, "descripcion": "string", "precio": number}]\n'
    '}\n'
    "<image>"
)

def run_inference(image_path: str) -> dict:
    result = model.infer(
        tokenizer,
        prompt=PROMPT,
        image_file=image_path,
        base_size=768,
        image_size=768,
        crop_mode=False,
        save_results=False,
    )
    return result

print("Función run_inference lista.")

## Celda 4 — Prueba con una imagen

In [ ]:
from IPython.display import display
from PIL import Image

# Cambia esta ruta a cualquier imagen de ticket que quieras probar
IMAGE_PATH = os.path.join(REPO_ROOT, "Deepseek OCR", "imagenes", "test.jpg")

display(Image.open(IMAGE_PATH).resize((400, 600)))

result = run_inference(IMAGE_PATH)
print(json.dumps(result, ensure_ascii=False, indent=2))

## Celda 5 — Evaluación batch

In [ ]:
import glob

# Directorio con imágenes de test
TEST_DIR = os.path.join(REPO_ROOT, "Deepseek OCR", "imagenes")
images   = glob.glob(os.path.join(TEST_DIR, "*.jpg")) + glob.glob(os.path.join(TEST_DIR, "*.png"))

print(f"Imágenes encontradas: {len(images)}")

results = []
for img_path in images:
    try:
        out = run_inference(img_path)
        valid = isinstance(out, dict) and "total" in out
        results.append({"image": os.path.basename(img_path), "valid": valid, "output": out})
        status = "OK" if valid else "MALFORMED"
        print(f"  [{status}] {os.path.basename(img_path)}")
    except Exception as e:
        results.append({"image": os.path.basename(img_path), "valid": False, "error": str(e)})
        print(f"  [ERROR] {os.path.basename(img_path)}: {e}")

valid_count = sum(1 for r in results if r["valid"])
print(f"\nResultados: {valid_count}/{len(results)} salidas válidas ({100*valid_count/max(len(results),1):.1f}%)")